# Clone-Censor-Weight Analysis with `ccw`

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yukiregista/clone-censor-weight/blob/main/examples/ccw_tutorial.ipynb)

This tutorial shows how to run a clone-censor-weight (CCW) analysis on
longitudinal
patient data with the `ccw` package. You will learn:

1. **Why** naive comparisons of "treated within X days vs. not" are biased, and how CCW fixes this.
2. **How** to fit a CCW analysis and estimate strategy-specific risks.
3. **How to check reliability** of the result, in two steps: the IPCW *tail-heaviness index* first,
   then the *bootstrap standard error*.

The whole notebook runs in about a minute.

## Why clone-censor-weight?

A common clinical question is: *does initiating a treatment within a few days of admission
improve outcomes, compared with not initiating it?* With observational data, simply comparing
"patients treated within the grace period" against "patients not treated" is biased in two ways:

- **Immortal time bias** — a patient must survive long enough to receive treatment.
  Patients who would have been treated but died first are misclassified as untreated.
- **Confounding by indication** — sicker patients are treated more (or less) often,
  so the groups are not comparable.

CCW analysis emulates a target trial in three steps:

1. **Clone** — at time zero, duplicate every eligible patient once per treatment strategy
   (e.g., *"initiate treatment by day 2"* vs. *"do not initiate through day 2"*).
2. **Censor** — artificially censor each clone at the moment the patient's observed data
   deviates from the clone's assigned strategy.
3. **Weight** — correct the selection bias introduced by this artificial censoring with
   inverse probability of censoring weights (IPCW), then estimate the risk under each
   strategy with a weighted Kaplan–Meier estimator.

The estimand is the per-protocol risk: what would have happened had everyone followed each strategy.

## Setup

In [ ]:
%pip install clone-censor-weight[research]

The `[research]` extra is only needed for the simulated dataset below (it ships the
data-generating code used in the accompanying paper). To analyze your own data,
`pip install ccw` alone is enough.

In [ ]:
import warnings

import numpy as np
import pandas as pd

import ccw

# statsmodels emits a noisy deprecation FutureWarning on every weight-model fit
warnings.filterwarnings("ignore", category=FutureWarning, module="statsmodels")

## A mock EHR cohort

We simulate a cohort of patients hospitalized with an acute respiratory infection,
using **the same data-generating process as Scenario 3 of the accompanying paper**
(the scenario with a time-varying covariate *and* informative pre-existing censoring):

| column | meaning |
|---|---|
| `id`, `time` | patient id and hospital day (0, 1, 2, ...) |
| `AGE`, `SEX_is_M`, `CCI` | baseline covariates (age, sex, Charlson Comorbidity Index) |
| `SPO2` | time-varying oxygen saturation, updated daily |
| `A` | treatment initiation (1 on the day treatment starts) |
| `D` | outcome: in-hospital death (1 on the day of death) |
| `CENS` | pre-existing censoring: discharge alive (informative — healthier patients leave earlier) |

Sicker patients (older, higher CCI, lower SpO₂) are more likely to be treated *and* more
likely to die — confounding by indication.

> **Using your own data:** replace this cell with your data loading. `ccw` expects the same
> *long format*: one row per patient per time unit, time starting at 0 and consecutive,
> with event columns coded 1 on the day the event occurs.

In [ ]:
from ccw._research.configs.load_variables import load_experiment_settings
from ccw._research.data_generation.core import BayesianNetwork
from ccw._research.utils import create_datasets_in_df

N_PATIENTS = 1000
FOLLOWUP_END = 31   # follow up through day 31; risks are reported at day 30
SEED = 42

params, variables, configs = load_experiment_settings("experimentD")
network = BayesianNetwork(variables, FOLLOWUP_END + 2)
sample = network.sample(sample_size=N_PATIENTS, seed=SEED)
_, _, _, data = create_datasets_in_df(
    network,
    sample,
    treatment_var=configs["treatment_var"],
    outcome_var=configs["outcome_var"],
    cut_data_after_outcome=configs["cut_data_after_outcome"],
    cutoff_time_of_observation=FOLLOWUP_END,
)
data, column_maps = configs["preprocess_pipeline"](data)

# CENS is generated as an absorbing state (stays 1 after discharge);
# ccw expects an *incident* indicator: 1 only on the day censoring occurs.
first_cens = data.loc[data["CENS"] == 1].groupby("id", sort=False)["time"].min()
data["CENS"] = (data["time"] == data["id"].map(first_cens)).astype(int)

print(f"{data['id'].nunique()} patients, {len(data)} patient-days")
print(f"deaths: {data.groupby('id')['D'].max().sum()}, "
      f"discharged alive: {data.groupby('id')['CENS'].max().sum()}")
data[["id", "time", "AGE", "SEX_is_M", "CCI", "SPO2", "A", "D", "CENS"]].head(8)

One preprocessing detail worth knowing: `SPO2` holds a normalized transform of the raw
saturation, as used by the paper's weight models (the original values are kept in
`SPO2_original`).

## The naive comparison (what *not* to do)

Compare 30-day mortality between patients who did vs. did not initiate treatment
within the grace period (days 0–2):

In [ ]:
GRACE = 2  # last day of the grace period

treated_by_grace = data.query("time <= @GRACE").groupby("id")["A"].max() == 1
died = data.groupby("id")["D"].max()
naive_risk = died.groupby(treated_by_grace).mean()
print(naive_risk.rename(index={True: "treated by day 2", False: "not treated"}))
print(f"naive risk difference: {naive_risk[True] - naive_risk[False]:+.3f}")

Treated patients die **more** often (risk difference ≈ +0.17) — treatment looks harmful.
This is exactly the pattern confounding by indication produces: treatment goes to the
patients who were already going to do worse. Let's see what CCW says.

## CCW analysis with the public API

Three objects describe the analysis:

- **`DataSpec`** maps your column names onto the roles `ccw` needs
  (id, time, treatment, outcome, censoring, covariates).
- **strategies** — here `InitiateBy(2)` ("initiate treatment by day 2") vs.
  `NoInitiationThrough(2)` ("do not initiate through day 2").
- **`weight_models`** — one IPCW formula per strategy *and* per censoring process.
  We model artificial censoring (deviation from the strategy) and pre-existing censoring
  (discharge alive) **separately** (`CensoringModel.SEPARATE`), because they arise from
  different mechanisms. Note the time term `C(tstart)` in the control model: under
  "do not initiate", deviation can occur on any day of the grace period, so the
  censoring hazard varies over time. Under "initiate by day 2", artificial censoring
  can only occur on day 2 itself, so no time term is needed. The `CENS` models carry
  no time term either, because in this simulation the discharge hazard is constant
  over time given the covariates — with real data, if pre-existing censoring plausibly
  depends on time, add `C(tstart)` to the `CENS` formulas as well.

In [ ]:
spec = ccw.DataSpec(
    id="id",
    time="time",
    treatment="A",
    outcome="D",
    censoring=("CENS",),
    baseline=("AGE", "SEX_is_M", "CCI"),
    time_varying=("SPO2",),
)

strategies = {
    "intervention": ccw.InitiateBy(GRACE),
    "control": ccw.NoInitiationThrough(GRACE),
}

weight_models = {
    "control": {
        "artificial_censor": "C(tstart) + AGE + SEX_is_M + CCI + SPO2",
        "CENS": "AGE + SEX_is_M + CCI",
    },
    "intervention": {
        "artificial_censor": "AGE + SEX_is_M + CCI + SPO2",
        "CENS": "AGE + SEX_is_M + CCI",
    },
}

analysis = ccw.CCW(
    spec=spec,
    strategies=strategies,
    weight_models=weight_models,
    followup_end=FOLLOWUP_END,
    estimate_at=30,
    censoring_model=ccw.CensoringModel.SEPARATE,
    # Declares that discharge alive cannot occur on day 0 (the admission day), so
    # day-0 rows are excluded from the risk set of the CENS weight model. False is
    # also the default; set True if censoring can occur at time zero in your data.
    censoring_at_baseline={"CENS": False},
)

result = analysis.fit(data)
result.summary()

CCW reverses the naive conclusion: the 30-day risk is *lower* under the intervention
strategy (risk difference ≈ −0.07). Before trusting this estimate — and before spending
time on bootstrap confidence intervals — check whether the weights can be trusted.

## Step 1 — the IPCW tail-heaviness index

IPCW asks uncensored patients to "stand in" for censored ones, so a few patients can
receive very large weights. The real danger is not one extreme weight in this dataset,
but a **heavy-tailed weight distribution**: then a handful of patients dominate the
estimate no matter how large the sample, and confidence intervals under-cover.

The **tail-heaviness index** α estimates how heavy that tail is (a weighted Hill
estimator applied to the inverse-probability factors at each grace-period day;
smaller α = heavier tail). Interpretation, following the paper:

- **α > 1** — valid large-sample inference is plausible → proceed to Step 2.
- **α ≤ 1** — warning: conventional and bootstrap confidence intervals may be invalid.
  Do not simply report a CI. First rule out model misspecification and data errors; if none,
  reconsider the eligibility criteria, strategy definitions, or grace period so that fewer
  patients have near-zero probability of adhering.

We summarize each strategy by the *minimum* index across grace-period days.
(For each day, the index is picked automatically from a stable region of the
Hill-estimator path; see the paper's supplementary material for details.)

In [ ]:
summary, _ = result.weight_diagnostics(
    patterns=("VAR",),   # the variance-relevant index reported in the paper
    max_time=GRACE,      # grace-period days only
    min_m=100,
)
summary[["a", "t", "pattern", "m", "tail_index", "status"]]

In [ ]:
tail_index = summary.query("status == 'ok'").groupby("a")["tail_index"].min()
print("tail-heaviness index (min over grace-period days):")
print(tail_index.round(2))

Both strategies have α > 1, so we proceed — but note the intervention arm sits close
to 1, an early hint that its weights are less well behaved than the control arm's.
(For comparison, in the paper's real-data analysis of 3,195 patients the indices were
12.7 and 8.5 — comfortably far from 1.)

## Step 2 — bootstrap standard errors

α > 1 says large-sample inference is *possible*; it does not say *this* sample is large
enough. Refit with patient-level bootstrap (clones of the same patient are resampled
together) and inspect the standard errors:

In [ ]:
analysis_boot = ccw.CCW(
    spec=spec,
    strategies=strategies,
    weight_models=weight_models,
    followup_end=FOLLOWUP_END,
    estimate_at=30,
    censoring_model=ccw.CensoringModel.SEPARATE,
    censoring_at_baseline={"CENS": False},
    n_bootstrap=100,      # use >=1000 for a real analysis; 100 keeps this cell quick
    bootstrap_seed=2025,
)
result_boot = analysis_boot.fit(data)
result_boot.summary()

In [ ]:
result_boot.bootstrap_results

In [ ]:
contrast = result_boot.contrast("intervention", "control")
rd, se = contrast.risk_difference, contrast.risk_difference_std_error
boot = result_boot.bootstrap_results
q_lo, q_hi = np.nanquantile(boot["intervention"] - boot["control"], [0.025, 0.975])
print(f"risk difference: {rd:+.3f}  (bootstrap SE {se:.3f})")
print(f"95% basic bootstrap CI: [{2*rd - q_hi:+.3f}, {2*rd - q_lo:+.3f}]")

The intervention-arm SE (≈ 0.09) is large, and the CI for the risk difference crosses zero.
In the paper's simulations, nominal 95% coverage was reached only once the estimator's
standard deviation fell to roughly 0.01 and was not reached at ≥ 0.10 — empirical reference
points, not universal cutoffs. Here the honest conclusion is: the point estimate favors
early treatment, the diagnostics permit inference, but with 1,000 patients the estimate
is too imprecise to be conclusive.

## Takeaways

For your own analysis, the workflow is:

1. Shape your data into long format and describe it with a `DataSpec`.
2. Fit `ccw.CCW` with per-strategy, per-process IPCW models → strategy-specific risks.
3. **Step 1:** `result.weight_diagnostics()` → tail-heaviness index per strategy.
   If α ≤ 1, stop and revisit the design (eligibility, strategies, grace period).
4. **Step 2:** if α > 1, refit with `n_bootstrap` and check that standard errors
   are small enough for your purpose before reporting confidence intervals.

**Further reading**

- Package documentation: <https://yukiregista.github.io/clone-censor-weight/>
- The accompanying paper (simulation evidence for the diagnostic procedure above,
  a real-world worked example, and a design/implementation checklist):
  *Pitfalls and Solutions in Clone-Censor-Weight for Target Trial Emulation* (Kimura, Takazawa, Yasunaga).